In [1]:
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

# Set random seed
SEED = 42

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# ImageNet normalization statistics
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Training Pipeline (32x32 native resolution - FAST on CPU)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation Pipeline (Fixed Evaluation View ONLY)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Datasets
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
val_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

# Optimized Loaders: batch_size=128 and num_workers=0 prevent CPU bottlenecks
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

num_classes = len(train_dataset.classes)
print(f"Data ready: {len(train_dataset)} training images, {len(val_dataset)} validation images across {num_classes} classes.")

Data ready: 50000 training images, 10000 validation images across 10 classes.


In [3]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

#### Feature Extraction

In [4]:
set_seed(42)

fe_model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in fe_model.parameters():
    param.requires_grad = False

fe_model.fc = nn.Linear(fe_model.fc.in_features, num_classes)
fe_model = fe_model.to(device)

trainable_params_fe = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)

optimizer_fe = optim.Adam(fe_model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 5
start_time = time.time()

print("--- Starting Feature Extraction Training ---")
for epoch in range(epochs):
    # Keep frozen backbone in eval() mode, train only the head
    fe_model.eval()
    fe_model.fc.train()
    
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_fe.zero_grad()
        outputs = fe_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_fe.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    val_acc_fe = evaluate(fe_model, val_loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc_fe:.4f}")

time_fe = round(time.time() - start_time, 2)
val_acc_fe = round(val_acc_fe, 4)

print(f"\n[Feature Extraction Complete] Total Time: {time_fe}s | Trainable Params: {trainable_params_fe}")

--- Starting Feature Extraction Training ---


C:\Users\harma\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/5 | Loss: 1.3767 | Val Acc: 0.5696
Epoch 2/5 | Loss: 1.1915 | Val Acc: 0.5953
Epoch 3/5 | Loss: 1.1540 | Val Acc: 0.6030
Epoch 4/5 | Loss: 1.1324 | Val Acc: 0.6119
Epoch 5/5 | Loss: 1.1185 | Val Acc: 0.6117

[Feature Extraction Complete] Total Time: 294.46s | Trainable Params: 5130


#### Fine Tuning

In [5]:
set_seed(42)

#Load model
ft_model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in ft_model.parameters():
    param.requires_grad = False

#Unfreeze backbone's layer4 & replace head
for param in ft_model.layer4.parameters():
    param.requires_grad = True

ft_model.fc = nn.Linear(ft_model.fc.in_features, num_classes)
ft_model = ft_model.to(device)

trainable_params_ft = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)

optimizer_ft = optim.Adam([
    {'params': ft_model.layer4.parameters(), 'lr': 0.0001},
    {'params': ft_model.fc.parameters(), 'lr': 0.001}
])

start_time = time.time()

for epoch in range(epochs):
    ft_model.train()  # Whole unfrozen model in train mode
    
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_ft.zero_grad()
        outputs = ft_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item() * images.size(0)
        
    val_acc_ft = evaluate(ft_model, val_loader)
    print(f"[Fine-Tuning] Epoch {epoch+1}/{epochs} - Val Acc: {val_acc_ft:.4f}")

time_ft = round(time.time() - start_time, 2)
val_acc_ft = round(val_acc_ft, 4)

[Fine-Tuning] Epoch 1/5 - Val Acc: 0.5411
[Fine-Tuning] Epoch 2/5 - Val Acc: 0.5915
[Fine-Tuning] Epoch 3/5 - Val Acc: 0.6181
[Fine-Tuning] Epoch 4/5 - Val Acc: 0.6353
[Fine-Tuning] Epoch 5/5 - Val Acc: 0.6427


In [6]:
print("=" * 65)
print(f"{'Stage':<25} | {'Val Accuracy':<12} | {'Train Time':<10} | {'Trainable Params':<15}")
print("-" * 65)
print(f"{'Feature Extraction':<25} | {val_acc_fe:<12} | {time_fe:<10}s | {trainable_params_fe:<15}")
print(f"{'Fine-Tuning':<25} | {val_acc_ft:<12} | {time_ft:<10}s | {trainable_params_ft:<15}")
print("=" * 65)

Stage                     | Val Accuracy | Train Time | Trainable Params
-----------------------------------------------------------------
Feature Extraction        | 0.6117       | 294.46    s | 5130           
Fine-Tuning               | 0.6427       | 629.54    s | 8398858        


#### Final Conclusion

Fine-tuning achieved higher validation accuracy than feature extraction.

Mechanism: Feature extraction relies on fixed ImageNet-trained representations. By unfreezing layer4 and training it with a smaller discriminative learning rate. Whereas, fine-tuning allowed the deep spatial feature filters to adapt specifically to CIFAR-10 characteristics while preserving early foundational representations.